# Interpolation of isochrone tables

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import json
import numpy as np
doc = json.loads(gravitas_input)
params = json.loads(gravitas_params) if gravitas_params else {}
assert doc.get("format") == "gravitas.observation" and doc.get("kind") == "table", "this notebook reads a gravitas.observation/1 table"
c = {k["id"]: np.array(k["values"], float) for k in doc["columns"]}
fehs = sorted(set(c["feh"].tolist()))
feh0 = min(fehs, key=lambda v: (abs(v), v))
req = params.get("request")
if req:
    assert float(req["feh"]) in fehs, "refused: the table does not hold [Fe/H] = %r" % (req["feh"],)
    sel_all = c["feh"] == float(req["feh"])
    assert c["logAge"][sel_all].min() <= float(req["logAge"]) <= c["logAge"][sel_all].max(), "refused: the age %r is outside the table (nothing is extrapolated)" % (req["logAge"],)
sel = (c["feh"] == feh0) & (c["phase"] == 0)
ages = sorted(set(c["logAge"][sel].tolist()))
per = {}
for a in ages:
    s = sel & (c["logAge"] == a)
    o = np.argsort(c["mass"][s])
    per[a] = (c["mass"][s][o], c["r"][s][o])
K = len(ages)
assert K >= 5, "refused: fewer than five ages"

In [ ]:
def r_at(a, mass):
    return float(np.interp(mass, per[a][0], per[a][1]))

def lagrange(xs, ys, x):
    total = 0.0
    for i in range(len(xs)):
        term = ys[i]
        for j in range(len(xs)):
            if j != i:
                term *= (x - xs[j]) / (xs[i] - xs[j])
        total += term
    return total

preds = []
for k in range(2, K - 2):
    nb = [ages[k - 2], ages[k - 1], ages[k + 1], ages[k + 2]]
    lo = max(per[a][0][0] for a in nb)
    hi = min(per[a][0][-1] for a in nb)
    for mass, rtrue in zip(*per[ages[k]]):
        if not (lo <= mass <= hi):
            continue
        ys = [r_at(a, mass) for a in nb]
        lin = ys[1] + (ys[2] - ys[1]) * (ages[k] - nb[1]) / (nb[2] - nb[1])
        preds.append([k, float(mass), float(rtrue), float(lin), float(lagrange(nb, ys, ages[k]))])
p = np.array(preds)
gravitas_output = {"answer": {"credit": doc.get("credit"), "license": doc["license"]["status"], "feh": feh0, "ages": ages, "count": len(preds), "predictions": preds,
    "rms": {"linear": float(np.sqrt(np.mean((p[:, 3] - p[:, 2]) ** 2))), "cubic": float(np.sqrt(np.mean((p[:, 4] - p[:, 2]) ** 2)))},
    "max": {"linear": float(np.max(np.abs(p[:, 3] - p[:, 2]))), "cubic": float(np.max(np.abs(p[:, 4] - p[:, 2])))}}}